### PageIndex RAG on local Ollama

In [1]:
import hashlib
import json
import os
import re
import tempfile
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import ollama

OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
HEAVY_MODEL = "granite4.2:8b"
GEN_MODEL = "qwen2.5:3b"
NUM_CTX = {HEAVY_MODEL: 65536, GEN_MODEL: 8192}  # heavy: largest node-search tree prompt is ~54k tokens
THINK_MODELS = {HEAVY_MODEL}
SEED = 0
KEEP_ALIVE = "30m"
RETRY_BASE_S = 2
CACHE_VERSION = 1
EVAL_WORKERS = 2
SMOKE_N = None

CACHE_DIR = Path("cache") / "ollama"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

client = ollama.Client(host=OLLAMA_HOST)
_lock = threading.Lock()
_think_rejected = set()
CALLS = {"total": 0}


def atomic_write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=path.parent, suffix=".tmp")
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump(obj, f, ensure_ascii=False)
        os.replace(tmp, path)
    except BaseException:
        if os.path.exists(tmp):
            os.remove(tmp)
        raise


def read_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        os.replace(path, str(path) + ".corrupt")
        return default


def strip_think(text):
    text = re.sub(r"<think>.*?</think>", "", text or "", flags=re.DOTALL)
    return text.split("<think>")[0].strip()


def parse_json(text):
    fence = "`" * 3
    text = strip_think(text).replace(fence + "json", "").replace(fence, "").strip()
    start, end = text.find("{"), text.rfind("}") + 1
    if start == -1 or end == 0:
        raise ValueError(f"no JSON object in: {text[:120]!r}")
    return json.loads(text[start:end])


def _chat_once(model, prompt, schema, num_predict):
    kwargs = dict(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0, "seed": SEED, "num_ctx": NUM_CTX[model], "num_predict": num_predict},
        keep_alive=KEEP_ALIVE,
    )
    if schema is not None:
        kwargs["format"] = schema
    if model in THINK_MODELS and model not in _think_rejected:
        kwargs["think"] = False
    try:
        resp = client.chat(**kwargs)
    except ollama.ResponseError as e:
        if "think" in kwargs and "think" in str(e).lower():
            _think_rejected.add(model)
            kwargs.pop("think")
            resp = client.chat(**kwargs)
        else:
            raise
    return resp.message.content or ""


def chat(model, prompt, *, schema=None, max_tokens=512, validate=None, retries=3):
    if len(prompt) / 3 + max_tokens > NUM_CTX[model]:
        print(f"WARNING: prompt may exceed num_ctx={NUM_CTX[model]} for {model}; Ollama would truncate it")
    last = None
    for attempt in range(retries):
        try:
            text = strip_think(_chat_once(model, prompt, schema, max_tokens * 2**attempt))
            if not text:
                raise ValueError("empty response")
            if validate:
                validate(text)
            with _lock:
                CALLS["total"] += 1
            return text
        except Exception as e:
            last = e
            time.sleep(RETRY_BASE_S * (attempt + 1))
    raise RuntimeError(f"{model} call failed after {retries} attempts: {last}")


def chat_json(model, prompt, schema, *, max_tokens=512):
    return parse_json(chat(model, prompt, schema=schema, max_tokens=max_tokens, validate=parse_json))


def stage_meta():
    return {"cache_version": CACHE_VERSION, "heavy": HEAVY_MODEL, "gen": GEN_MODEL, "num_ctx": NUM_CTX}


def load_stage(name):
    path = CACHE_DIR / f"{name}.json"
    data = read_json(path)
    if isinstance(data, dict) and data.get("meta") == stage_meta():
        return data
    if data is not None:
        os.replace(path, CACHE_DIR / f"{name}.stale.json")
    return {"meta": stage_meta(), "items": {}}


def save_stage(name, data):
    with _lock:
        atomic_write_json(CACHE_DIR / f"{name}.json", data)


def run_parallel(tasks, fn, workers, on_done):
    ex = ThreadPoolExecutor(max_workers=workers)
    try:
        futures = {ex.submit(fn, t): t for t in tasks}
        for f in as_completed(futures):
            on_done(futures[f], f.result())
    finally:
        ex.shutdown(wait=True, cancel_futures=True)


def check_models(required):
    try:
        names = {m.model for m in client.list().models}
    except Exception as e:
        raise RuntimeError(f"Cannot reach Ollama at {OLLAMA_HOST}. Start it with `ollama serve`. ({e})") from e
    for m in required:
        if m not in names and f"{m}:latest" not in names:
            raise RuntimeError(f"Model {m} not installed. Run: ollama pull {m}")

In [2]:
PDF_SOURCE_DIR = "data"
QA_FILE = Path("data") / "policy_qa.json"
PAGEINDEX_STORE = ".pageindex"  # reuse the already-built tree store instead of rebuilding
REGISTRY_PATH = str(CACHE_DIR / "pdf_registry.json")
MODEL_STR = f"ollama_chat/{HEAVY_MODEL}"
SUMMARY_CONCURRENCY = 1
SUFFIX = "_smoke" if SMOKE_N else ""
OUTPUTS_STAGE = "stage_pageindex_outputs" + SUFFIX
EVAL_STAGE = "stage_pageindex_eval" + SUFFIX
os.environ.setdefault("OLLAMA_API_BASE", OLLAMA_HOST)

check_models([HEAVY_MODEL, GEN_MODEL])
_probe = client.chat(
    model=HEAVY_MODEL,
    messages=[{"role": "user", "content": "Say ok"}],
    think=False,
    options={"num_predict": 8, "num_ctx": NUM_CTX[HEAVY_MODEL]},
    keep_alive=KEEP_ALIVE,
)
assert not getattr(_probe.message, "thinking", None), "granite is still thinking; update Ollama"
print(f"Ollama ready | heavy={HEAVY_MODEL} gen={GEN_MODEL} | smoke_n={SMOKE_N} | eval_workers={EVAL_WORKERS}")

Ollama ready | heavy=granite4.2:8b gen=qwen2.5:3b | smoke_n=None | eval_workers=2


### Running PageIndex Locally

In [3]:
import asyncio

import pageindex.utils as _u
from pageindex import PageIndexLocalClient, utils

_u.SUMMARY_CONCURRENCY = SUMMARY_CONCURRENCY


async def _capped(model, prompt):
    if len(prompt) > 6000:
        prompt = prompt[:6000] + "\n...[truncated]"
    return await asyncio.to_thread(chat, HEAVY_MODEL, prompt)


_u.llm_acompletion = _capped

if hasattr(_u, "llm_completion"):
    # classic PageIndex pipeline: the sync helper takes chat_history, which is flattened into one prompt
    def _completion(model, prompt, chat_history=None, return_finish_reason=False, *args, **kwargs):
        if chat_history:
            history = "\n\n".join(f"{m['role']}: {m['content']}" for m in chat_history)
            prompt = f"{history}\n\nuser: {prompt}"
        text = chat(HEAVY_MODEL, prompt, max_tokens=2048)
        return (text, "finished") if return_finish_reason else text

    _u.llm_completion = _completion

pi_client = PageIndexLocalClient(
    model=MODEL_STR,
    summary_model=MODEL_STR,
    retrieve_model=MODEL_STR,
    storage_path=PAGEINDEX_STORE,
)
print(pi_client.__class__.__name__)

PageIndexLocalClient


### Building The JSON Tree

In [4]:
pdf_files = sorted(
    str(p) for p in Path(PDF_SOURCE_DIR).glob("**/*.pdf")
)
print(f"Found {len(pdf_files)} PDFs")

existing = {d["name"]: d["id"] for d in pi_client.list_documents()["documents"]}
doc_ids = {}
failed = []
consecutive = 0
for pdf_path in pdf_files:
    filename = os.path.basename(pdf_path)
    if filename in existing:
        doc_ids[filename] = existing[filename]
        print(f"Already indexed: {filename}")
        continue
    t0 = time.time()
    try:
        doc_ids[filename] = pi_client.submit_document(pdf_path)["doc_id"]  # synchronous
        print(f"Submitted: {filename} -> {doc_ids[filename]} ({time.time() - t0:.0f}s)")
        consecutive = 0
    except Exception as e:
        failed.append(filename)
        consecutive += 1
        print(f"Failed: {filename}: {e}")
        if consecutive >= 3:
            raise RuntimeError("3 PDFs failed in a row: is Ollama still running? Progress is saved; re-run this cell.")

print(f"\nIndexed {len(doc_ids)}/{len(pdf_files)} PDFs")
if failed:
    raise RuntimeError(f"{len(failed)} PDFs not indexed: {failed}. Re-run this cell.")

Found 15 PDFs
Already indexed: 14..Trade credit insc_GEN756.pdf
Already indexed: Arogya_Sanjeevani_Policy_Wording_KEN_034037d936.pdf
Already indexed: Auto_Secure_Commercial_Vehicle_Package_Policy_Base_Policy_Wording_22b7905015.pdf
Already indexed: Bharat_Griha_Raksha_Policy_Policy_Wordings_5219f40e18.pdf
Already indexed: Click-2-Protect-Optima-Secure-Policy-Bond-101Y122V05.pdf
Already indexed: Cyber_Shield_Policy_Wordings_78baa23b5a.pdf
Already indexed: PORT_PACKAGE_Policy_wording_a5f317091b.pdf
Already indexed: Policy_Wordings_aviation_insurance.pdf_7b7e60200a.pdf
Already indexed: Policy_Wordings_contractors_plant_and_machinery_insurance.pdf_0175bcd067.pdf
Already indexed: Policy_Wordings_political_risk_insurance_for_investors.pdf_b7a0e7c805.pdf
Already indexed: SBI_General_Livestock_Policy_Wording_38ef0b201b.pdf
Already indexed: Weather_Insurance_Policy_Wordings_Retail_6a1bd806a7.pdf
Already indexed: hdfc-life-smart-pension-plus-v13-policy-document-individual.pdf
Already indexed: spe

In [5]:
print(json.dumps(doc_ids, indent=4))

{
    "14..Trade credit insc_GEN756.pdf": "pi-58068b94fd464bedaf251e3b442d3183",
    "Arogya_Sanjeevani_Policy_Wording_KEN_034037d936.pdf": "pi-7a77210c58344134988c513ce697a93a",
    "Auto_Secure_Commercial_Vehicle_Package_Policy_Base_Policy_Wording_22b7905015.pdf": "pi-66af3e82285e4e24819dcc87fcda1343",
    "Bharat_Griha_Raksha_Policy_Policy_Wordings_5219f40e18.pdf": "pi-a47f79df4ac040f5984d512c4e1225ae",
    "Click-2-Protect-Optima-Secure-Policy-Bond-101Y122V05.pdf": "pi-12de45f887e644009982f039d294eb7e",
    "Cyber_Shield_Policy_Wordings_78baa23b5a.pdf": "pi-6392aba4cb2f45ce8abf2f75871f3e93",
    "PORT_PACKAGE_Policy_wording_a5f317091b.pdf": "pi-da50dbffde964da4ba36ede6c7079e11",
    "Policy_Wordings_aviation_insurance.pdf_7b7e60200a.pdf": "pi-155f96f798b24b41982378c2bef596a5",
    "Policy_Wordings_contractors_plant_and_machinery_insurance.pdf_0175bcd067.pdf": "pi-9d8ead02fe1f4f6284671bf58c07e80b",
    "Policy_Wordings_political_risk_insurance_for_investors.pdf_b7a0e7c805.pdf": "pi-

In [6]:
d = pi_client.list_documents()["documents"][0]["id"]
t = pi_client.get_tree(d, node_summary=True)["result"]
print(len(t), "top-level nodes")
print(utils.create_node_mapping(t).keys().__len__(), "total nodes")

sizes = []
for doc in pi_client.list_documents()["documents"]:
    tree = pi_client.get_tree(doc["id"], node_summary=True)["result"]
    tree_json = json.dumps(utils.remove_fields(tree.copy(), fields=["text"]), indent=2)
    sizes.append((int(len(tree_json) / 3), doc["name"]))
largest = max(sizes)
print(f"Largest node-search prompt: ~{largest[0]} tokens ({largest[1]}); NUM_CTX[HEAVY_MODEL] = {NUM_CTX[HEAVY_MODEL]}")
if largest[0] + 1000 > NUM_CTX[HEAVY_MODEL]:
    print("WARNING: raise NUM_CTX[HEAVY_MODEL] (for example 32768), or Ollama truncates that prompt silently")

66 top-level nodes
72 total nodes
Largest node-search prompt: ~53759 tokens (PORT_PACKAGE_Policy_wording_a5f317091b.pdf); NUM_CTX[HEAVY_MODEL] = 65536


### Creating PDF registry for routing

In [7]:
registry = {
    d["id"]: {"doc_id": d["id"], "filename": d["name"], "description": d.get("description", "")}
    for d in pi_client.list_documents()["documents"]
}
atomic_write_json(REGISTRY_PATH, registry)
print(f"Registry written: {len(registry)} docs, 0 Ollama calls")

Registry written: 15 docs, 0 Ollama calls


### Query Router

In [8]:
ROUTE_SCHEMA = {
    "type": "object",
    "properties": {
        "thinking": {"type": "string"},
        "type": {"type": "string", "enum": ["general", "specific", "ambiguous"]},
        "doc_ids": {"type": "array", "items": {"type": "string"}},
        "clarification": {"type": "string"},
    },
    "required": ["thinking", "type", "doc_ids", "clarification"],
}


def route_query(query: str) -> dict:
    with open(REGISTRY_PATH, "r", encoding="utf-8") as f:
        registry = json.load(f)

    catalog = [
        {
            "doc_id": v["doc_id"],
            "filename": v["filename"],
            "description": v["description"],
        }
        for v in registry.values()
    ]

    prompt = f"""You are a document router for an insurance Q&A assistant.
A user has asked a question. Decide which documents to search and classify the question type.

User question: {query}

Available documents:
{json.dumps(catalog, indent=2)}

Classify the question into ONE of these types:
- "general"    : conceptual, definitional, or comparative questions ("what is X?", "explain Y", "how does Z work", "difference between A and B"). Answerable from general knowledge — EVEN IF one of the documents happens to cover that topic.
- "specific"   : the user asks about a concrete provision, condition, claim, or coverage detail of a policy — the kind of thing you'd only find in the actual wording (grace periods, exclusions, claim process, deductibles, coverage limits).
- "ambiguous"  : too vague to route (e.g. "tell me about my policy").

Reply ONLY with this JSON, nothing else:
{{
    "thinking": "<one short sentence, max ~20 words>",
    "type": "specific",
    "doc_ids": ["doc_id_1"],
    "clarification": ""
}}

Rules:
- type "general"   → doc_ids must be [], clarification must be ""
- type "ambiguous" → doc_ids must be [], clarification must be a follow-up question
- type "specific"  → clarification must be ""
- Rule of thumb: if the answer wouldn't change between documents, it's "general". Only search documents when the user needs THEIR policy's actual terms.
- For "specific": list at most the 3 most relevant doc_ids, never more.
"""

    parsed = chat_json(HEAVY_MODEL, prompt, ROUTE_SCHEMA, max_tokens=512)
    parsed["doc_ids"] = [d for d in parsed.get("doc_ids", []) if d in registry][:3]

    print(f"Type     : {parsed['type']}")
    print(f"Doc IDs  : {parsed['doc_ids']}")
    print(f"Reasoning: {parsed['thinking']}")

    return parsed

In [9]:
# # Test all three types
# route_query("What happens if I miss a premium payment?")  # should be specific
# route_query("What is term insurance?")  # should be general
# route_query("Tell me about my policy")  # should be ambiguous

### Search nodes in relevant documents

In [10]:
NODE_SCHEMA = {
    "type": "object",
    "properties": {
        "thinking": {"type": "string"},
        "node_list": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["thinking", "node_list"],
}


def search_nodes(doc_id: str, query: str) -> list[str]:
    """
    Searches the node tree of a single PDF for content relevant to the query.
    Returns a list of text chunks tagged with source filename and page number.
    """
    if not pi_client.is_retrieval_ready(doc_id):
        print(f"Doc {doc_id} not ready - skipping.")
        return []

    with open(REGISTRY_PATH, "r", encoding="utf-8") as f:
        registry = json.load(f)

    filename = registry[doc_id]["filename"]

    # fetch tree for this specific doc
    tree = pi_client.get_tree(doc_id, node_summary=True)["result"]
    tree_without_text = utils.remove_fields(tree.copy(), fields=["text"])

    prompt = f"""You are given a tree structure of a document, then a question.
Each node contains a node id, title, and summary.
Find all nodes likely to contain the answer to the question.

Document tree:
{json.dumps(tree_without_text, indent=2)}

Question: {query}

Reply ONLY with this JSON:
{{
    "thinking": "<one short sentence, max ~20 words>",
    "node_list": ["node_id_1", "node_id_2"]
}}
"""

    result = chat_json(HEAVY_MODEL, prompt, NODE_SCHEMA, max_tokens=512)

    node_map = utils.create_node_mapping(tree)

    chunks = []
    for node_id in result["node_list"]:
        if node_id not in node_map:
            continue
        node = node_map[node_id]
        chunks.append(
            f"[Source: {filename}, Page {node['page_index']}]\n{node['text'][:4000]}"
        )

    print(f"  {filename}: {len(chunks)} relevant node(s) found")
    return chunks


In [11]:
def ask(query: str):
    print(f"\n{'=' * 60}")
    print(f"Query: {query}")
    print("=" * 60)

    # Step 1: route — classify query and get relevant doc_ids
    routing = route_query(query)
    q_type = routing["type"]

    # Step 2: branch based on question type
    if q_type == "ambiguous":
        print(f"\nCould you clarify: {routing['clarification']}")
        return

    elif q_type == "general":
        print("\nGeneral question — answering from LLM knowledge.\n")
        prompt = f"""Answer this insurance question in simple plain language.
        Start with a one-sentence summary. Avoid jargon.
        Question: {query}"""
        answer = chat(GEN_MODEL, prompt, max_tokens=1024)
        utils.print_wrapped(answer)

    elif q_type == "specific":
        print(f"\nSearching {len(routing['doc_ids'])} document(s)...")

        # Step 3: search nodes in each routed doc and collect chunks
        all_chunks = []
        for doc_id in routing["doc_ids"]:
            chunks = search_nodes(doc_id, query)
            all_chunks.extend(chunks)

        if not all_chunks:
            print("No relevant content found.")
            return

        context = "\n\n---\n\n".join(all_chunks)

        # Step 4: generate answer from retrieved context
        prompt = f"""Answer the question based only on the context below.
If context comes from multiple documents, mention which document each point is from.

Question: {query}

Context:
{context}

Instructions:
- Use plain simple language, avoid legal jargon
- Use "you" and "your" instead of "the policyholder"
- Start with a one-sentence summary
- End with "Bottom line:" telling the user what to actually do or know
"""
        answer = chat(GEN_MODEL, prompt, max_tokens=1024)
        print("\nAnswer:\n")
        utils.print_wrapped(answer)

In [12]:
# ask("What happens if I miss a premium payment?")

In [13]:
# ask("What is the SECTION V Of CYBER VAULTEDGE policy wording?")

### Test set

In [14]:
QA_ALIASES = {
    "question": ("question", "query", "q"),
    "ground_truth": ("ground_truth", "answer", "ground_truth_answer", "reference", "expected_answer"),
    "source_file": ("source_file", "source", "pdf", "document", "file", "filename"),
    "page": ("page", "page_number", "page_no"),
}


def load_qa(path=QA_FILE) -> list[dict]:
    raw = json.loads(Path(path).read_text(encoding="utf-8"))
    if isinstance(raw, dict):
        raw = next((v for v in raw.values() if isinstance(v, list)), [])
    if raw:
        print(f"{path}: {len(raw)} items; keys of the first item: {sorted(raw[0])}")
    rows = []
    for n, item in enumerate(raw):
        row = dict(item)
        for key, aliases in QA_ALIASES.items():
            found = next((a for a in aliases if item.get(a) not in (None, "")), None)
            row[key] = (item[found] if key == "page" else str(item[found])) if found else ("" if key != "page" else None)
        if not row["question"] or not row["ground_truth"]:
            raise KeyError(f"item {n} has no question or ground truth; its keys: {sorted(item)}")
        rows.append(row)
    return rows


test_set = load_qa()
if len(test_set) != 83:
    print(f"WARNING: expected 83 questions, found {len(test_set)}")
if SMOKE_N:
    test_set = test_set[:SMOKE_N]
    print(f"SMOKE_N={SMOKE_N}: running the first {len(test_set)} questions only")

data/policy_qa.json: 83 items; keys of the first item: ['answer', 'category', 'chunk_id', 'documents', 'filename', 'id', 'question', 'source_title']


In [15]:
# Preview the test set
print(f"Total questions: {len(test_set)}\n")
for i, item in enumerate(test_set[:5], 1):
    print(f"Q{i}: {item['question']}")
    print(f"   GT: {item['ground_truth'][:120]}...")
    print(f"   From: {item['source_file']} p.{item['page']}")
    print()

Total questions: 83

Q1: What consequence does the policy impose if the insured fails to provide the required documents in accordance with Item 12.2?
   GT: The insurer may give the insured notice suspending all cover under the policy until the insured fully complies with the ...
   From: 14..Trade credit insc_GEN756.pdf p.None

Q2: Which countries are excluded from coverage or require advance approval, thereby voiding a claim under this policy?
   GT: The policy excludes Somalia, Afghanistan, and Palestine (off‑cover) and requires advance insurer approval for Iran,...
   From: 14..Trade credit insc_GEN756.pdf p.None

Q3: What exchange rate is used to calculate the amount of indemnification under this Trade Credit Insurance policy?
   GT: The rate of exchange used by the Reserve Bank of India on the day the Insured Event occurred....
   From: 14..Trade credit insc_GEN756.pdf p.None

Q4: How does the policy describe the process for filing a claim and the documents required?
   GT: The p

### Retrieval and answer generation

In [16]:
TREE_CACHE = {}
NODE_MAP_CACHE = {}
REGISTRY_CACHE = None


def load_registry() -> dict:
    global REGISTRY_CACHE
    if REGISTRY_CACHE is None:
        with open(REGISTRY_PATH, encoding="utf-8") as f:
            REGISTRY_CACHE = json.load(f)
    return REGISTRY_CACHE


def _pdf_name(name: str) -> str:
    base = os.path.basename(name).lower()
    return base if base.endswith(".pdf") else base + ".pdf"


def doc_id_for_eval_item(item: dict, registry: dict) -> str | None:
    """Eval rows already know their source PDF, so avoid an LLM router call."""
    source_file = item.get("source_file", "")
    if not source_file:
        return None
    for doc_id, meta in registry.items():
        if _pdf_name(meta["filename"]) == _pdf_name(source_file):
            return doc_id
    return None


def get_tree_and_node_map(doc_id: str):
    if doc_id not in TREE_CACHE:
        tree = pi_client.get_tree(doc_id, node_summary=True)["result"]
        TREE_CACHE[doc_id] = tree
        NODE_MAP_CACHE[doc_id] = utils.create_node_mapping(tree)
    return TREE_CACHE[doc_id], NODE_MAP_CACHE[doc_id]


def search_nodes_cached(doc_id: str, query: str, max_chunks: int = 2) -> list[str]:
    """Same PageIndex tree selection as search_nodes(), but caches local tree work."""
    if not pi_client.is_retrieval_ready(doc_id):
        print(f"  Doc {doc_id} not ready - skipping.")
        return []

    registry = load_registry()
    filename = registry[doc_id]["filename"]
    tree, node_map = get_tree_and_node_map(doc_id)
    tree_without_text = utils.remove_fields(tree.copy(), fields=["text"])

    prompt = f"""You are given a tree structure of a document, then a question.
Each node contains a node id, title, and summary.
Find up to {max_chunks} nodes likely to contain the answer to the question.

Document tree:
{json.dumps(tree_without_text, indent=2)}

Question: {query}

Reply ONLY with this JSON:
{{
    "thinking": "<one short sentence, max ~20 words>",
    "node_list": ["node_id_1", "node_id_2"]
}}
"""

    result = chat_json(HEAVY_MODEL, prompt, NODE_SCHEMA, max_tokens=512)

    chunks = []
    for node_id in result.get("node_list", [])[:max_chunks]:
        node = node_map.get(node_id)
        if not node:
            continue
        chunks.append(
            f"[Source: {filename}, Page {node['page_index']}]\n{node['text'][:3000]}"
        )

    print(f"  {filename}: {len(chunks)} relevant node(s) found")
    return chunks


def run_pageindex_pipeline(
    test_set: list[dict],
    start: int = 0,
    limit: int | None = None,
    max_chunks: int = 2,
) -> list[dict]:
    """
    - uses each test row's source_file instead of the LLM router when it is known
    - searches only that PDF, matching ask()'s specific-document path
    - caches PageIndex trees/node maps
    - checkpoints after every question; a failed question stays unsaved and is retried on the next run
    """
    registry = load_registry()
    end = len(test_set) if limit is None else min(start + limit, len(test_set))
    stage = load_stage(OUTPUTS_STAGE)
    failures = 0

    print(f"Running PageIndex RAG on questions {start + 1}-{end} of {len(test_set)}...\n")

    for i in range(start, end):
        item = test_set[i]
        q = item["question"]
        if str(i) in stage["items"]:
            continue
        print(f"[{i + 1}/{len(test_set)}] {q[:70]}...")
        t0 = time.perf_counter()

        try:
            doc_id = doc_id_for_eval_item(item, registry)
            q_type = "eval_source_file"

            if doc_id is None:
                routing = route_query(q)
                q_type = routing["type"]
                doc_ids_to_search = routing.get("doc_ids", [])[:1]
            else:
                doc_ids_to_search = [doc_id]

            all_chunks = []
            for doc_id_to_search in doc_ids_to_search:
                all_chunks.extend(
                    search_nodes_cached(doc_id_to_search, q, max_chunks=max_chunks)
                )

            if all_chunks:
                context = "\n\n---\n\n".join(all_chunks)
                answer = chat(
                    GEN_MODEL,
                    f"""Answer the question based only on the context below.
Question: {q}

Context:
{context}

Instructions:
- Use plain simple language
- Start with a one-sentence summary
- End with "Bottom line:" telling the user what to know""",
                    max_tokens=384,
                )
            else:
                answer = "No relevant content found."

            stage["items"][str(i)] = {
                "id": i,
                "question": q,
                "ground_truth": item["ground_truth"],
                "contexts": all_chunks,
                "answer": answer,
                "q_type": q_type,
                "source_file": item["source_file"],
                "doc_ids_searched": doc_ids_to_search,
            }
            save_stage(OUTPUTS_STAGE, stage)
            failures = 0
            print(f"  OK {len(all_chunks)} chunks | {time.perf_counter() - t0:.0f}s | Answer: {answer[:60]}...")

        except Exception as e:
            failures += 1
            print(f"  FAILED: {e}")
            if failures >= 3:
                raise RuntimeError("3 questions failed in a row: is Ollama still running? Progress is saved; re-run this cell.")

    missing = [i + 1 for i in range(start, end) if str(i) not in stage["items"]]
    if missing:
        raise RuntimeError(f"{len(missing)} questions unfinished: Q{missing}. Re-run this cell.")
    print(f"\nDone: {end - start}/{end - start} questions")
    return [stage["items"][str(i)] for i in range(start, end)]


EVAL_BATCH_START = 0
EVAL_BATCH_SIZE = None
EVAL_MAX_CHUNKS = 2

pageindex_outputs = run_pageindex_pipeline(
    test_set,
    start=EVAL_BATCH_START,
    limit=EVAL_BATCH_SIZE,
    max_chunks=EVAL_MAX_CHUNKS,
)
atomic_write_json(CACHE_DIR / f"pageindex_outputs{SUFFIX}.json", pageindex_outputs)
print(f"Saved PageIndex outputs to {CACHE_DIR / f'pageindex_outputs{SUFFIX}.json'}")

Running PageIndex RAG on questions 1-83 of 83...


Done: 83/83 questions
Saved PageIndex outputs to cache/ollama/pageindex_outputs.json


### Evaluation prompts

In [17]:
FAITHFULNESS_PROMPT = """You are an expert evaluator assessing whether an AI answer is faithful to its source context.

QUESTION: {question}

RETRIEVED CONTEXT:
{context}

GENERATED ANSWER:
{answer}

TASK:
1. List every factual claim made in the Generated Answer.
2. For each claim, determine if it is directly supported by the Retrieved Context.
3. Score = (number of supported claims) / (total claims). If there are no claims, score 1.0.
Keep "reasoning" terse — a few words per claim, one line total, no more than ~60 words overall.

Respond in this EXACT JSON format only (no markdown, no extra text):
{{"score": 0.0, "reasoning": "claim 1: supported/not supported because... claim 2: ..."}}

Score must be between 0.0 and 1.0."""


ANSWER_RELEVANCY_PROMPT = """You are an expert evaluator assessing whether an AI answer is relevant to the question asked.

QUESTION: {question}

GENERATED ANSWER:
{answer}

TASK:
Score how directly and completely the answer addresses the question.
- 1.0 = answer directly addresses all parts of the question
- 0.7 = answer mostly relevant but misses a part or adds off-topic content
- 0.4 = answer is vaguely related but does not really answer the question
- 0.0 = answer is completely off-topic or refuses to answer
Keep "reasoning" to one short sentence.

Respond in this EXACT JSON format only:
{{"score": 0.0, "reasoning": "explanation of why this score was given"}}"""


CONTEXT_PRECISION_PROMPT = """You are an expert evaluator assessing the quality of retrieved context for a RAG system.

QUESTION: {question}

GROUND TRUTH ANSWER:
{ground_truth}

RETRIEVED CONTEXT CHUNKS:
{context_numbered}

TASK:
For each retrieved chunk, decide if it is relevant to answering the question (given what the ground truth says).
Score = (number of relevant chunks) / (total chunks).
If no chunks were retrieved, score is 0.0.
Keep "reasoning" terse — a few words per chunk, one line total, no more than ~60 words overall.

Respond in this EXACT JSON format only:
{{"score": 0.0, "reasoning": "Chunk 1: relevant/not relevant because... Chunk 2: ..."}}"""


CONTEXT_RECALL_PROMPT = """You are an expert evaluator assessing whether a RAG system retrieved all necessary information.

QUESTION: {question}

GROUND TRUTH ANSWER:
{ground_truth}

RETRIEVED CONTEXT:
{context}

TASK:
1. List every key piece of information in the Ground Truth Answer.
2. For each key piece, check if it is present in the Retrieved Context.
3. Score = (pieces present in context) / (total key pieces).
If no context was retrieved, score is 0.0.
Keep "reasoning" terse — a few words per point, one line total, no more than ~60 words overall.

Respond in this EXACT JSON format only:
{{"score": 0.0, "reasoning": "Key point 1: found/not found in context... Key point 2: ..."}}"""

print("Evaluation prompts defined.")

Evaluation prompts defined.


### Judge and parallel evaluator

In [18]:
METRICS = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
JUDGE_MAX_TOKENS = 512

SCORE_SCHEMA = {
    "type": "object",
    "properties": {"reasoning": {"type": "string"}, "score": {"type": "number"}},
    "required": ["reasoning", "score"],
}


def safe_parse_score(raw: str) -> dict:
    """Parses the judge's JSON and clamps the score to [0, 1]. Raises on malformed output."""
    parsed = parse_json(raw)
    score = max(0.0, min(1.0, float(parsed["score"])))
    return {"score": score, "reasoning": parsed.get("reasoning", "")}


def judge_prompt(metric: str, item: dict) -> str:
    q, a, gt = item["question"], item["answer"], item["ground_truth"]
    ctx = item.get("contexts", [])
    context_joined = "\n\n".join(ctx) if ctx else "[No context retrieved]"
    context_numbered = (
        "\n\n".join(f"[Chunk {i + 1}]:\n{c}" for i, c in enumerate(ctx))
        if ctx
        else "[No context retrieved]"
    )
    if metric == "faithfulness":
        return FAITHFULNESS_PROMPT.format(question=q, context=context_joined, answer=a)
    if metric == "answer_relevancy":
        return ANSWER_RELEVANCY_PROMPT.format(question=q, answer=a)
    if metric == "context_precision":
        return CONTEXT_PRECISION_PROMPT.format(
            question=q, ground_truth=gt, context_numbered=context_numbered
        )
    return CONTEXT_RECALL_PROMPT.format(question=q, ground_truth=gt, context=context_joined)


def judge_metric(metric: str, item: dict) -> dict:
    text = chat(
        HEAVY_MODEL,
        judge_prompt(metric, item),
        schema=SCORE_SCHEMA,
        max_tokens=JUDGE_MAX_TOKENS,
        validate=safe_parse_score,
    )
    return safe_parse_score(text)


def evaluate_single(item: dict) -> dict:
    """Evaluates one (question, contexts, answer, ground_truth) entry on all four metrics."""
    return {m: judge_metric(m, item) for m in METRICS}


def _input_hash(item: dict) -> str:
    payload = json.dumps(
        [item["question"], item["answer"], item["ground_truth"], item["contexts"]],
        ensure_ascii=False,
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:16]


def evaluate_pipeline(pipeline_outputs: list[dict], strategy_name: str) -> list[dict]:
    """
    Evaluates all outputs of one RAG strategy with EVAL_WORKERS parallel judge calls.
    One task is one (question, metric) pair; the stage file is saved after each task.
    """
    total = len(pipeline_outputs)
    stage = load_stage(EVAL_STAGE)
    hashes = {o["id"]: _input_hash(o) for o in pipeline_outputs}

    def is_done(o, m):
        r = stage["items"].get(f"{o['id']}:{m}")
        return r is not None and r.get("h") == hashes[o["id"]]

    tasks = [(o, m) for o in pipeline_outputs for m in METRICS if not is_done(o, m)]
    print(f"\n{'=' * 60}")
    print(f"Evaluating: {strategy_name} ({total} questions x 4 metrics = {total * 4} judge calls)")
    print(f"{len(tasks)} to do, {total * 4 - len(tasks)} already checkpointed, workers={EVAL_WORKERS}")
    print(f"{'=' * 60}")

    finished = 0
    t0 = time.perf_counter()

    def judge(task):
        o, m = task
        return judge_metric(m, o)

    def done(task, result):
        nonlocal finished
        o, m = task
        stage["items"][f"{o['id']}:{m}"] = {**result, "h": hashes[o["id"]]}
        save_stage(EVAL_STAGE, stage)
        if all(is_done(o, k) for k in METRICS):
            finished += 1
            s = {k: stage["items"][f"{o['id']}:{k}"]["score"] for k in METRICS}
            print(
                f"  Q{o['id'] + 1} "
                f"F={s['faithfulness']:.2f}  AR={s['answer_relevancy']:.2f}  "
                f"CP={s['context_precision']:.2f}  CR={s['context_recall']:.2f}  "
                f"[{time.perf_counter() - t0:.0f}s]"
            )

    run_parallel(tasks, judge, EVAL_WORKERS, done)

    missing = [(o["id"] + 1, m) for o in pipeline_outputs for m in METRICS if not is_done(o, m)]
    if missing:
        raise RuntimeError(f"{len(missing)} judge tasks unfinished. Re-run this cell.")

    results = []
    for o in pipeline_outputs:
        scores = {
            m: {k: stage["items"][f"{o['id']}:{m}"][k] for k in ("score", "reasoning")}
            for m in METRICS
        }
        results.append(
            {
                "question": o["question"],
                "answer": o["answer"],
                "ground_truth": o["ground_truth"],
                "n_contexts": len(o.get("contexts", [])),
                "faithfulness": scores["faithfulness"]["score"],
                "answer_relevancy": scores["answer_relevancy"]["score"],
                "context_precision": scores["context_precision"]["score"],
                "context_recall": scores["context_recall"]["score"],
                "reasoning": scores,
            }
        )
    print(f"\nDone: {len(results)}/{total} questions evaluated")
    return results


print("Evaluator functions defined.")

Evaluator functions defined.


### Evaluate PageIndex RAG

In [19]:
pageindex_eval_results = evaluate_pipeline(pageindex_outputs, "PageIndex RAG")
atomic_write_json(CACHE_DIR / f"pageindex_eval_results{SUFFIX}.json", pageindex_eval_results)
print(f"Saved to {CACHE_DIR / f'pageindex_eval_results{SUFFIX}.json'}")

means = {m: sum(r[m] for r in pageindex_eval_results) / len(pageindex_eval_results) for m in METRICS}
print("Mean scores:", {k: round(v, 3) for k, v in means.items()})
print(f"Ollama calls this session: {CALLS['total']}")


Evaluating: PageIndex RAG (83 questions x 4 metrics = 332 judge calls)
272 to do, 60 already checkpointed, workers=2


  Q15 F=0.00  AR=0.40  CP=0.50  CR=0.00  [3s]
  Q16 F=1.00  AR=0.40  CP=0.50  CR=1.00  [20s]
  Q17 F=1.00  AR=0.70  CP=0.00  CR=0.00  [47s]
  Q18 F=0.00  AR=1.00  CP=0.00  CR=0.00  [68s]
  Q19 F=0.75  AR=0.40  CP=0.00  CR=0.00  [96s]
  Q20 F=0.50  AR=0.70  CP=1.00  CR=1.00  [120s]
  Q21 F=1.00  AR=0.40  CP=1.00  CR=1.00  [150s]
  Q22 F=0.25  AR=0.40  CP=0.00  CR=0.00  [179s]
  Q23 F=1.00  AR=1.00  CP=0.50  CR=1.00  [198s]
  Q24 F=0.00  AR=0.40  CP=0.00  CR=0.00  [223s]
  Q25 F=1.00  AR=0.40  CP=0.50  CR=0.00  [264s]
  Q26 F=0.00  AR=0.70  CP=0.00  CR=0.00  [287s]
  Q27 F=1.00  AR=0.40  CP=0.50  CR=1.00  [316s]
  Q28 F=1.00  AR=0.40  CP=0.50  CR=0.00  [343s]
  Q29 F=0.00  AR=0.40  CP=0.00  CR=0.00  [366s]
  Q30 F=1.00  AR=0.40  CP=0.50  CR=0.00  [397s]
  Q31 F=1.00  AR=1.00  CP=0.50  CR=1.00  [423s]
  Q32 F=0.00  AR=1.00  CP=0.00  CR=0.00  [448s]
  Q33 F=0.88  AR=0.40  CP=0.50  CR=0.80  [482s]
  Q34 F=0.00  AR=0.40  CP=0.50  CR=1.00  [511s]
  Q35 F=0.00  AR=0.40  CP=1.00  CR=1.00  [537s

F = Faithfulness

AR = Answer Relevancy

CP = Context Precision

CR = Context Recall